# Rössler System — PE vs ETC Analysis

Compares Permutation Entropy (PE) and Effort-To-Compress (ETC) against the Maximum Lyapunov Exponent (MLE) across a sweep of `c` (controls transition to chaos, a=b=0.2 fixed).

**Fixed parameters:** a = 0.2, b = 0.2

**MLE units:** nats/time-unit

**Contents**
1. Setup & data loading
2. Noise-free analysis — PE and ETC vs `c`, overlaid with MLE
3. Noise robustness — effect of measurement noise on PE and ETC
4. Correlation vs σ — quantitative robustness summary
5. Summary statistics table

## 1. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# ── Shared plot style ────────────────────────────────────────────────────────
import matplotlib
matplotlib.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
    "axes.titlesize": 11,
    "legend.fontsize": 9,
    "text.usetex": True,
    "font.family": "serif",
})
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
def savefig(fig, fig_dir, name):
    for ext in ("png", "pdf"):
        fig.savefig(fig_dir / f"{name}.{ext}", dpi=300, bbox_inches="tight")
    print(f"  Saved → {fig_dir.name}/{name}.png/.pdf")
def plot_correlation_vs_sigma(data, L, D_array, bin_array, noise_levels, fig_dir, map_label):
    """
    One figure: Pearson correlation of PE/ETC with MLE vs noise sigma.
    PE lines: one per D value.
    ETC lines: one per bins value.
    Separate figures per L.
    """
    mle = data['mle']
    valid = np.isfinite(mle)

    fig, ax = plt.subplots(figsize=(8, 5))

    pe_cmap = matplotlib.colormaps['Greens'].resampled(len(D_array) + 2)
    etc_cmap = matplotlib.colormaps['Blues'].resampled(len(bin_array) + 2)

    for j, D in enumerate(D_array):
        corrs = []
        for sigma in noise_levels:
            pe_vals = data['pe'][sigma][D]
            mask = valid & np.isfinite(pe_vals)
            corrs.append(np.corrcoef(mle[mask], pe_vals[mask])[0, 1])
        ax.plot(noise_levels, corrs, 'o-',
                color=pe_cmap(j + 2), linewidth=2, markersize=5,
                label=f"PE  D={D}")

    for j, bins in enumerate(bin_array):
        corrs = []
        for sigma in noise_levels:
            etc_vals = data['etc'][sigma][bins]
            mask = valid & np.isfinite(etc_vals)
            corrs.append(np.corrcoef(mle[mask], etc_vals[mask])[0, 1])
        ax.plot(noise_levels, corrs, 's--',
                color=etc_cmap(j + 2), linewidth=2, markersize=5,
                label=f"ETC bins={bins}")

    ax.set(xlabel=r"Noise $\sigma$",
           ylabel=r"Pearson correlation with MLE",
           title=f"{map_label} : Correlation vs Noise  (L={L:,})",
           ylim=[None, 1.00])
    ax.legend(ncol=2, fontsize=9)
    plt.tight_layout()
    savefig(fig, fig_dir, f"correlation_vs_sigma_L{L}")
    plt.show()

from src.io_utils import load_results

FIG_DIR = ROOT / 'figures' / 'rossler'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete. Figures →', FIG_DIR)

## Configuration

**Edit this cell** to point to your data files and select which `L`, `D`, and `bins` values to display.

In [ ]:
# ── Point to your data files ────────────────────────────────────────────
DATA_DIR = ROOT / 'data'

# List of L values to analyse — one figure set is produced per L
L_VALUES = [100, 1000, 10000, 100000, 1000000]

# Which D and bins values to include in the combined figures
D_ARRAY   = [5, 6, 7]
BIN_ARRAY = [2, 4, 6]#, 8]

# Noise levels to show in the robustness plots
# (must match what was generated by the script)
NOISE_LEVELS = [0.0, 0.01, 0.05, 0.1]

# Label for titles and filenames
MAP_LABEL = 'Rössler System'

print('Config loaded. L_VALUES =', L_VALUES)

# Parameter axis label (used in plot titles)
PARAM_NAME = 'c'

## Load data

Loads all requested L values into a dict `ALL_DATA = {L: data_dict}`.

In [ ]:
ALL_DATA = {}
for L in L_VALUES:
    fpath = DATA_DIR / f'rossler_L{L}.h5'
    if not fpath.exists():
        print(f'WARNING: {fpath} not found — skipping L={L}')
        continue
    ALL_DATA[L] = load_results(str(fpath))
    print(f'Loaded L={L}: {len(ALL_DATA[L]["noise_levels"])} noise levels, '
          f'PE dims={sorted(ALL_DATA[L]["pe"][0.0].keys())}, '
          f'ETC bins={sorted(ALL_DATA[L]["etc"][0.0].keys())}')

print(f'\nLoaded {len(ALL_DATA)} dataset(s).')

## 2. Noise-free analysis

PE and ETC vs `c` at σ=0, overlaid with MLE.
One figure per L value; `D` / `bins` values as subplot columns.

The MLE trace is plotted on a shared secondary y-axis (right) in grey, with a dashed horizontal line at MLE = 0 marking the chaos boundary.

In [ ]:
for L, data in ALL_DATA.items():
    param  = data['param_values']
    mle    = data['mle']
    sigma0 = 0.0

    nD = len(D_ARRAY)
    nB = len(BIN_ARRAY)
    nCols = max(nD, nB)

    fig, axes = plt.subplots(2, nCols,
                             figsize=(5 * nCols, 8),
                             sharey=False)
    # If only one column, wrap so indexing is always axes[row][col]
    if nCols == 1:
        axes = [[axes[0]], [axes[1]]]

    # ── Row 0: PE subplots ───────────────────────────────────────────────────
    for j, D in enumerate(D_ARRAY):
        ax  = axes[0][j]
        ax2 = ax.twinx()

        pe_vals = data['pe'][sigma0][D]
        ax.plot(param, pe_vals, color='tab:green', lw=1.5, label=f'PE D={D}')
        ax2.plot(param, mle, color='grey', lw=1.0, alpha=0.5, label='MLE')
        ax2.axhline(0, color='grey', lw=0.8, ls='--')

        ax.set_xlabel(PARAM_NAME)
        ax.set_ylabel('PE (normalised)', color='tab:green')
        ax2.set_ylabel('MLE', color='grey')
        ax.set_title(f'PE  D={D}')
        ax.tick_params(axis='y', labelcolor='tab:green')
        ax2.tick_params(axis='y', labelcolor='grey')

    # Hide unused columns in row 0
    for j in range(nD, nCols):
        axes[0][j].set_visible(False)

    # ── Row 1: ETC subplots ──────────────────────────────────────────────────
    for j, bins in enumerate(BIN_ARRAY):
        ax  = axes[1][j]
        ax2 = ax.twinx()

        etc_vals = data['etc'][sigma0][bins]
        ax.plot(param, etc_vals, color='tab:blue', lw=1.5, label=f'ETC bins={bins}')
        ax2.plot(param, mle, color='grey', lw=1.0, alpha=0.5, label='MLE')
        ax2.axhline(0, color='grey', lw=0.8, ls='--')

        ax.set_xlabel(PARAM_NAME)
        ax.set_ylabel('ETC (NETC1D)', color='tab:blue')
        ax2.set_ylabel('MLE', color='grey')
        ax.set_title(f'ETC  bins={bins}')
        ax.tick_params(axis='y', labelcolor='tab:blue')
        ax2.tick_params(axis='y', labelcolor='grey')

    # Hide unused columns in row 1
    for j in range(nB, nCols):
        axes[1][j].set_visible(False)

    fig.suptitle(f'{MAP_LABEL} — PE & ETC vs {PARAM_NAME} (noise-free, L={L:,})', y=1.01)
    plt.tight_layout()
    savefig(fig, FIG_DIR, f'noisefree_L{L}')
    plt.show()

## 3. Noise robustness

All noise levels overlaid on the same axis (different colours) for each D / bins column.
MLE is shown once as a grey background trace (using σ=0 data).

In [ ]:
noise_cmap = matplotlib.colormaps['YlOrRd'].resampled(len(NOISE_LEVELS) + 2)

for L, data in ALL_DATA.items():
    param = data['param_values']
    mle   = data['mle']

    nD  = len(D_ARRAY)
    fig, axes = plt.subplots(1, nD, figsize=(5*nD, 4), sharey=False)
    if nD == 1:
        axes = [axes]

    for j, D in enumerate(D_ARRAY):
        ax  = axes[j]
        ax2 = ax.twinx()
        ax2.plot(param, mle, color='grey', lw=0.8, alpha=0.4)
        ax2.axhline(0, color='grey', lw=0.8, ls='--')
        ax2.set_ylabel('MLE', color='grey')
        ax2.tick_params(axis='y', labelcolor='grey')

        for k, sigma in enumerate(NOISE_LEVELS):
            if sigma not in data['pe']:
                continue
            pe_vals = data['pe'][sigma][D]
            label   = f'σ={sigma}' if sigma > 0 else 'clean'
            ax.plot(param, pe_vals,
                    color=noise_cmap(k + 1), lw=1.2,
                    alpha=0.85, label=label)

        ax.set_xlabel('c')
        ax.set_ylabel('PE (normalised)', color='tab:green')
        ax.tick_params(axis='y', labelcolor='tab:green')
        ax.set_title(f'PE  D={D}')
        ax.legend(fontsize=8)

    fig.suptitle(f'{MAP_LABEL} — PE noise robustness  (L={L:,})', y=1.02)
    plt.tight_layout()
    savefig(fig, FIG_DIR, f'pe_noise_L{L}')
    plt.show()

In [ ]:
for L, data in ALL_DATA.items():
    param = data['param_values']
    mle   = data['mle']

    nB  = len(BIN_ARRAY)
    fig, axes = plt.subplots(1, nB, figsize=(5*nB, 4), sharey=False)
    if nB == 1:
        axes = [axes]

    for j, bins in enumerate(BIN_ARRAY):
        ax  = axes[j]
        ax2 = ax.twinx()
        ax2.plot(param, mle, color='grey', lw=0.8, alpha=0.4)
        ax2.axhline(0, color='grey', lw=0.8, ls='--')
        ax2.set_ylabel('MLE', color='grey')
        ax2.tick_params(axis='y', labelcolor='grey')

        for k, sigma in enumerate(NOISE_LEVELS):
            if sigma not in data['etc']:
                continue
            etc_vals = data['etc'][sigma][bins]
            label    = f'σ={sigma}' if sigma > 0 else 'clean'
            ax.plot(param, etc_vals,
                    color=noise_cmap(k + 1), lw=1.2,
                    alpha=0.85, label=label)

        ax.set_xlabel('c')
        ax.set_ylabel('ETC (NETC1D)', color='tab:blue')
        ax.tick_params(axis='y', labelcolor='tab:blue')
        ax.set_title(f'ETC  bins={bins}')
        ax.legend(fontsize=8)

    fig.suptitle(f'{MAP_LABEL} — ETC noise robustness  (L={L:,})', y=1.02)
    plt.tight_layout()
    savefig(fig, FIG_DIR, f'etc_noise_L{L}')
    plt.show()

## 4. Correlation vs σ

Pearson correlation between each measure and MLE, plotted as a function of noise σ.  A robust measure should maintain high correlation even at large σ.  Separate figure per L.

In [ ]:
for L, data in ALL_DATA.items():
    plot_correlation_vs_sigma(
        data, L, D_ARRAY, BIN_ARRAY,
        NOISE_LEVELS, FIG_DIR, MAP_LABEL
    )

## 5. Summary statistics table

Pearson correlation of each measure with MLE, at every `(L, D/bins, σ)` combination.  Values close to 1 indicate the measure tracks MLE well.

In [ ]:
import pandas as pd

rows = []
for L, data in ALL_DATA.items():
    mle   = data['mle']
    valid = np.isfinite(mle)
    for sigma in NOISE_LEVELS:
        if sigma not in data['pe']:
            continue
        for D in D_ARRAY:
            pe_vals = data['pe'][sigma][D]
            mask = valid & np.isfinite(pe_vals)
            r = np.corrcoef(mle[mask], pe_vals[mask])[0, 1]
            rows.append({'L': L, 'Measure': f'PE D={D}',
                         'sigma': sigma, 'Pearson_r': round(r, 4)})
        for bins in BIN_ARRAY:
            etc_vals = data['etc'][sigma][bins]
            mask = valid & np.isfinite(etc_vals)
            r = np.corrcoef(mle[mask], etc_vals[mask])[0, 1]
            rows.append({'L': L, 'Measure': f'ETC bins={bins}',
                         'sigma': sigma, 'Pearson_r': round(r, 4)})

df = pd.DataFrame(rows)
pivot = df.pivot_table(index=['L', 'Measure'],
                       columns='sigma',
                       values='Pearson_r')
pivot.columns = [f'σ={c}' for c in pivot.columns]
print(pivot.to_string())

# Save as CSV alongside figures
csv_path = FIG_DIR / f'{MAP_LABEL}_correlation_table.csv'
pivot.to_csv(csv_path)
print(f'\nSaved → {csv_path}')